# Mobile Money Agent Network — Data Cleanup & Audit

## 1. Import Pipeline & Clean Data

In [ ]:
import pandas as pd
from cleaning_pipeline import clean_agent_data

txns = pd.read_csv('agent_transactions.csv')
agents = pd.read_csv('agents.csv')

clean_df, log = clean_agent_data(txns, agents)
print('Raw rows:', log['raw_txn_rows'], '| Clean rows:', log['final_clean_rows'])
display(clean_df.head())

## 2. Post-Merge Validation

In [ ]:
print('Raw transaction rows:', len(txns))
print('Post-merge clean rows:', len(clean_df))
print('Unknown agent transactions:', int((clean_df['county'] == 'Unknown').sum()))
print('Unique txn_id count:', clean_df['txn_id'].nunique())
print('txn_type values:', sorted(clean_df['txn_type'].unique().tolist()))

## 3. County Summary & Weekly Pivot Table

In [ ]:
county_summary = (
    clean_df.groupby('county')
    .agg(transactions=('txn_id', 'count'), total_amount=('amount', 'sum'), avg_amount=('amount', 'mean'), outliers=('is_outlier_iqr', 'sum'))
    .sort_values('total_amount', ascending=False)
)
print('=== County Summary ===')
display(county_summary)

clean_df['week'] = pd.to_datetime(clean_df['txn_date']).dt.to_period('W-MON').astype(str)
weekly_pivot = clean_df.pivot_table(index='week', columns='txn_type', values='amount', aggfunc='sum', fill_value=0.0)
print('=== Weekly Pivot Table (Amount by Transaction Type) ===')
display(weekly_pivot)

## 4. Export Clean Transactions (`outputs/agent_transactions_clean.csv`)

In [ ]:
import os
os.makedirs('outputs', exist_ok=True)
cols_out = [c for c in clean_df.columns if c not in ['txn_date_dt', 'week']]
clean_df[cols_out].to_csv('outputs/agent_transactions_clean.csv', index=False)
print('Saved outputs/agent_transactions_clean.csv with shape:', clean_df[cols_out].shape)

## Six-Dimension Data Quality Report

- **Completeness:** Raw transactions contained 20 missing or non-numeric (`N/A`) amount entries; all rows were preserved and flagged as missing or invalid.
- **Uniqueness:** Duplicate records (11 rows removed) were eliminated so `txn_id` is unique in `outputs/agent_transactions_clean.csv`.
- **Consistency:** Inconsistent casing and trailing whitespace in `txn_type` were standardised strictly to `Deposit` and `Withdrawal`.
- **Validity:** Mixed date delimiters (`/` and `-`) were parsed and formatted into ISO `YYYY-MM-DD`, while `amount` was cast to numeric.
- **Accuracy:** IQR outlier detection and high-value checks flagged 40 outlier transactions (`is_outlier_iqr = True`) for review.
- **Timeliness:** Left-joining `agent_transactions` to `agents` reconciled 601 records across the March–April 2026 window and identified 8 transactions with unknown agents (`AG099`) missing from the register.